In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from dataclasses import dataclass
import joblib

In [19]:
@dataclass
class args:
    BATCH_SIZE: int = 4
    NUM_WORKERS:int = 0
    RANDOM_SEED: int = 74
    filename:str = './medical-o1-reasoning-SFT.pkl'

In [20]:
args = args()


In [21]:
torch.set_float32_matmul_precision('high')
# Setting the seed
pl.seed_everything(args.RANDOM_SEED)
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
# print("Device:", device)


Seed set to 74


In [22]:
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_devices}")
    for i in range(num_devices):
        device_name = torch.cuda.get_device_name(i)
        print(f"Device {i}: {device_name}")
else:
    print("No CUDA devices available.")

Number of CUDA devices: 2
Device 0: NVIDIA GeForce RTX 4060 Ti
Device 1: NVIDIA GeForce RTX 4060


In [31]:
class ClassDataset(Dataset):
    def __init__(self):
        self.data =  joblib.load(args.filename, mmap_mode='r')
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return {
            'input_ids': self.data[index]["input_ids"],
            'attention_mask': self.data[index]["attention_mask"],
            'labels': self.data[index]["labels"],
        }

In [32]:
datasetLoad = ClassDataset()
datasetLoader = DataLoader(
        dataset=datasetLoad, 
        batch_size = args.BATCH_SIZE, 
        shuffle=True, 
        num_workers= args.NUM_WORKERS,
        pin_memory=True
    )

In [33]:
enum_list = enumerate(datasetLoader)
print(next(enum_list))

(0, {'input_ids': tensor([[128000, 128256,    198,  ..., 128001, 128001, 128001],
        [128000, 128256,    198,  ..., 128001, 128001, 128001],
        [128000, 128256,    198,  ..., 128001, 128001, 128001],
        [128000, 128256,    198,  ..., 128001, 128001, 128001]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[  -100,   -100,   -100,  ..., 128001, 128001, 128001],
        [  -100,   -100,   -100,  ..., 128001, 128001, 128001],
        [  -100,   -100,   -100,  ..., 128001, 128001, 128001],
        [  -100,   -100,   -100,  ..., 128001, 128001, 128001]])})
